In [5]:
# =============================================
# 🚨 IMPORTANT: INSTALL THE RIGHT LIBRARIES FIRST
# =============================================
# Before running, make sure you have installed the necessary packages with GPU support.
# Run this in your terminal if you haven't already:
# CMAKE_ARGS="-DLLAMA_CUBLAS=on" FORCE_CMAKE=1 pip install --upgrade --force-reinstall llama-cpp-python --no-cache-dir
# pip install -q langchain langchain-community
# =============================================

import os
from datetime import datetime
from langchain.agents import initialize_agent, AgentType, load_tools, Tool
from langchain_community.llms import LlamaCpp # Or any other LLM you are using

# --- 1. Initialize the LLM ---
# Make sure the model path is correct for your system
model_path = r"D:\NetShieldAI_Chatbot\pretrained_language_model\openhermes-2.5-mistral-7b.Q4_K_M.gguf"

# Check if the model file exists before trying to load it
if not os.path.exists(model_path):
    print(f"Error: Model file not found at path: {model_path}")
    print("Please update the 'model_path' variable with the correct location of your GGUF model.")
    llm = None
else:
    print("Loading the local GGUF model...")
    llm = LlamaCpp(
        model_path=model_path,
        n_ctx=2048,           # Context window size
        n_gpu_layers=10,      # Number of layers to offload to GPU (adjust based on your VRAM)
        temperature=0.1,      # Lower for more deterministic agent behavior
        verbose=False,        # Set to False to keep the output clean during inference
    )
    print("Model loaded successfully!")

# --- 2. Load In-Built Tools ---
# We'll load the 'llm-math' tool. The agent will use this for calculations.
print("Loading in-built tools...")
in_built_tools = load_tools(["llm-math"], llm=llm)

# --- 3. Create a Custom Tool ---
# A simple function to get the current time.
def get_current_time(input: str = "") -> str:
    """
    Returns the current date and time.
    Use this tool whenever you need to know the current time or date.
    The input argument is ignored, so you can pass an empty string.
    """
    print(">> Custom Tool 'get_current_time' was called!")
    return datetime.now().strftime("%A, %B %d, %Y %I:%M:%S %p")

# Wrap the Python function in LangChain's 'Tool' class.
# The 'description' is CRITICAL. The agent reads this to decide WHEN to use the tool.
print("Creating custom tools...")
custom_time_tool = Tool.from_function(
    func=get_current_time,
    name="Current Time Checker",
    description="Useful for getting the current date and time. The Action Input for this tool should always be an empty string.",
)

# --- 4. Combine All Tools ---
# Create a single list containing all the tools the agent can use.
tools = in_built_tools + [custom_time_tool]

# --- 5. Initialize the ReAct Agent ---
# We provide the tools, the LLM, the agent type, and set verbose=True to see its thoughts.
# The `handle_parsing_errors=True` parameter is added to fix the previous error.
if llm:
    agent = initialize_agent(
        tools=tools,
        llm=llm,
        agent=AgentType.ZERO_SHOT_REACT_DESCRIPTION,
        verbose=True, # Set to True to see the agent's reasoning process!
        handle_parsing_errors=True  # ✅ This is the key fix.
    )

    # --- 6. Test the Agent ---
    print("\n--- Testing the agent with a multi-tool query ---")
    query = "What time is it right now, and what is 76 divided by 3?"
    
    try:
        response = agent.run(query)
        print("\n✅ Final Answer:")
        print(response)
    except Exception as e:
        print(f"\nAn error occurred during agent execution: {e}")

else:
    print("\nSkipping agent execution because the LLM could not be loaded.")

Loading the local GGUF model...


llama_context: n_batch is less than GGML_KQ_MASK_PAD - increasing to 64
llama_context: n_ctx_per_seq (2048) < n_ctx_train (32768) -- the full capacity of the model will not be utilized


Model loaded successfully!
Loading in-built tools...
Creating custom tools...

--- Testing the agent with a multi-tool query ---


> Entering new AgentExecutor chain...
 I need to find out the current time and also calculate 76 divided by 3.
Action: Current Time Checker
Action Input: ''>> Custom Tool 'get_current_time' was called!

Observation: Tuesday, October 07, 2025 05:10:26 PM
Thought: I have the current time. Now, I need to calculate 76 divided by 3.
Action: Calculator
Action Input: 76/3
Observation: Answer: 25.333333333333332
Thought: I have the result of the calculation.
Final Answer: The current time is Tuesday, October 07, 2025 05:10:26 PM and 76 divided by 3 is approximately 25.333333333333332.

> Finished chain.

✅ Final Answer:
The current time is Tuesday, October 07, 2025 05:10:26 PM and 76 divided by 3 is approximately 25.333333333333332.


In [7]:
from langchain.memory import ConversationBufferMemory

# In a real app, this would be a database. For this example, a simple global list is perfect.
TODO_LIST = []

def add_task(task: str) -> str:
    """
    Adds a task to the to-do list.
    Use this tool to add a new item to the user's list of tasks.
    The input should be the task description.
    """
    print(f">> Custom Tool 'add_task' was called with task: {task}")
    TODO_LIST.append(task)
    return f"Successfully added '{task}' to the to-do list."

def view_tasks(input: str = "") -> str:
    """
    Views all tasks in the to-do list.
    Use this tool when the user wants to see what is currently on their list.
    This tool does not require any input.
    """
    print(">> Custom Tool 'view_tasks' was called!")
    if not TODO_LIST:
        return "The to-do list is currently empty."
    
    # Format the list for the agent
    tasks = "\n".join(f"- {item}" for item in TODO_LIST)
    return f"Here are the items on your to-do list:\n{tasks}"

# Create the Tool objects for the agent
add_task_tool = Tool.from_function(
    func=add_task,
    name="ToDo Adder",
    description="Use this to add a task to a to-do list. Input should be the task to add.",
)

view_tasks_tool = Tool.from_function(
    func=view_tasks,
    name="ToDo Viewer",
    description="Use this to view all tasks on the to-do list. It takes no input.",
)

# Combine the tools into a list for the agent
tools = [add_task_tool, view_tasks_tool]

# --- Initialize Memory ---
memory = ConversationBufferMemory(
    memory_key="chat_history",
    return_messages=True
)

# --- Initialize the Conversational Agent ---
if llm:
    agent = initialize_agent(
        tools=tools,
        llm=llm,
        agent=AgentType.CONVERSATIONAL_REACT_DESCRIPTION,
        memory=memory,
        verbose=True,
        handle_parsing_errors=True
    )

    # ---  Test the Multi-Turn To-Do List Flow ---
    print("\n--- Turn 1: Introduction ---")
    response1 = agent.run("Hi, my name is Sam.")
    print("\n✅ Final Answer (Turn 1):")
    print(response1)

    print("\n" + "="*50 + "\n")

    print("\n--- Turn 2: Adding a task ---")
    response2 = agent.run("Please add 'buy groceries' to my to-do list.")
    print("\n✅ Final Answer (Turn 2):")
    print(response2)

    print("\n" + "="*50 + "\n")
    
    print("\n--- Turn 3: Adding another task ---")
    response3 = agent.run("Great, also add 'finish the weekly report'.")
    print("\n✅ Final Answer (Turn 3):")
    print(response3)

    print("\n" + "="*50 + "\n")
    
    print("\n--- Turn 4: Viewing the list and testing memory ---")
    response4 = agent.run("What's on my list now? Also, what is my name?")
    print("\n✅ Final Answer (Turn 4):")
    print(response4)

else:
    print("\nSkipping agent execution because the LLM could not be loaded.")

C:\Users\Divy\AppData\Local\Temp\ipykernel_6332\1571229130.py:47: LangChainDeprecationWarning: Please see the migration guide at: https://python.langchain.com/docs/versions/migrating_memory/
  memory = ConversationBufferMemory(



--- Turn 1: Introduction ---


> Entering new AgentExecutor chain...

Thought: Do I need to use a tool? No
AI: Hello Sam! It's nice to meet you. How can I assist you today?

> Finished chain.

✅ Final Answer (Turn 1):
Hello Sam! It's nice to meet you. How can I assist you today?



--- Turn 2: Adding a task ---


> Entering new AgentExecutor chain...

Response:
```
Thought: Do I need to use a tool? Yes
Action: ToDo Adder
Action Input: buy groceries>> Custom Tool 'add_task' was called with task: buy groceries

Observation: Successfully added 'buy groceries' to the to-do list.
Thought: Do I need to use a tool? No
AI: The task 'buy groceries' has been added to your to-do list. Is there anything else you would like me to assist with?
```

> Finished chain.

✅ Final Answer (Turn 2):
The task 'buy groceries' has been added to your to-do list. Is there anything else you would like me to assist with?
```



--- Turn 3: Adding another task ---


> Entering new AgentExecutor chain...

Thought: 